# LLM zero-shot / few-shot on LIAR (Qwen2.5-7B, 4-bit)
Runtime > Change runtime type > T4 GPU, then Run all (~25 min).

In [ ]:
# Step 1: check which GPU Colab gave us (this run needs a T4 or better).
!nvidia-smi -L
# Step 2: download the project code into a folder called "repo".
!git clone https://github.com/arunchowdary-1/Fake-News-Detection-Fine-tuned-Transformers-vs-Zero-shot-LLMs.git repo
# Move into the repo so the python -m commands below can find the llm/ and common/ packages.
# (Keep comments on their own line: a trailing # after %cd would be read as part of the folder name.)
%cd repo
# Step 3: install what Colab does not have. bitsandbytes = 4-bit loading, accelerate = places the model on the GPU.
# 'datasets<4' is pinned because datasets 4 no longer supports the script-based LIAR dataset.
!pip install -q -U bitsandbytes accelerate transformers 'datasets<4' scikit-learn

In [ ]:
# Smoke test: 20 statements, writes llm_zeroshot_limit.json only.
# It checks that the model loads and that the answers parse into labels. It does not touch the real results files.
# --4bit loads Qwen2.5-7B-Instruct in 4-bit (FP4) so it fits on a 16 GB T4; --limit 20 stops after 20 statements.
!python -m llm.evaluate_llm --4bit --mode zeroshot --limit 20

In [ ]:
# Zero-shot run on the shared 300-statement test subset (instruction only, no examples).
# Writes results/llm_zeroshot.json and results/llm_zeroshot_preds.csv.
# Takes several minutes: the 4-bit model is loaded first, then each statement is classified one at a time (batch size 1).
!python -m llm.evaluate_llm --4bit --mode zeroshot

In [ ]:
# Few-shot run on the same 300 statements: the prompt also carries 6 worked examples (one per class) from the training split.
# Writes results/llm_fewshot.json and results/llm_fewshot_preds.csv. Slower than zero-shot because every request is longer.
!python -m llm.evaluate_llm --4bit --mode fewshot

In [ ]:
# Build the cost-vs-accuracy comparison from every results/*.json that exists.
# Writes results/cost_comparison.json, results/cost_comparison.md and results/cost_vs_accuracy.csv.
# Prices every model at a T4 rate of $0.35/hour plus 10% overhead.
!python -m llm.cost_analysis

In [ ]:
# Save the results. The Google Drive copy is optional: edit DEST to a real folder, or delete the cp line.
# (With the CHANGE_ME placeholder left in, the cp command fails; the zip download below still works.)
from google.colab import drive, files
drive.mount("/content/drive")
DEST = "/content/drive/MyDrive/CHANGE_ME_shared_folder/"
# Copy the two LLM result files, their prediction CSVs and the cost files into the Drive folder.
!cp results/llm_zeroshot.json results/llm_fewshot.json results/llm_zeroshot_preds.csv results/llm_fewshot_preds.csv results/cost_* "{DEST}"
# Also bundle the same files into one zip and download it to your computer.
!zip -q llm_results.zip results/llm_*.json results/llm_*_preds.csv results/cost_*
files.download("llm_results.zip")